# การสร้างแอปพลิเคชันการสร้างภาพ (Azure OpenAI)

LLMs ไม่ได้มีแค่การสร้างข้อความเท่านั้น คุณยังสามารถสร้างภาพจากคำอธิบายข้อความได้ ภาพในฐานะของสื่อชนิดหนึ่งมีประโยชน์ในหลายสาขา เช่น เทคโนโลยีการแพทย์ สถาปัตยกรรม การท่องเที่ยว การพัฒนาเกม การตลาด และอื่น ๆ ในบทเรียนนี้เราจะทำงานกับโมเดล **GPT Image** รุ่นปัจจุบันบน Microsoft Foundry

## เป้าหมายการเรียนรู้

เมื่อจบบทเรียนนี้ คุณจะสามารถ:

- อธิบายว่าการสร้างภาพคืออะไรและใช้ประโยชน์ได้ที่ไหน
- เข้าใจครอบครัวโมเดล `gpt-image` และความแตกต่างจากโมเดล DALL·E รุ่นเก่า
- สร้างแอปพลิเคชันการสร้างภาพและแก้ไขภาพ

## การสร้างภาพคืออะไร?

โมเดลการสร้างภาพจะสร้างภาพจากข้อความคำสั่ง โมเดลสมัยใหม่เช่น `gpt-image` จะเรียนรู้ความสัมพันธ์ระหว่างข้อความกับภาพในระหว่างการฝึกฝน แล้วค่อย ๆ เปลี่ยนแปลงเสียงรบกวนสุ่มให้กลายเป็นภาพที่ตรงกับคำสั่งของคุณ

สองตระกูลโมเดลภาพที่เป็นที่รู้จักก็คือ:

- **`gpt-image` (OpenAI)** — เจเนอเรชันปัจจุบันที่ใช้ในบทเรียนนี้ รองรับการสร้างภาพจากข้อความและการแก้ไขภาพ (การเติมภาพด้วยมาสก์)
- **Midjourney** — โมเดลของบุคคลที่สามที่ได้รับความนิยม มีบริการและกระบวนการทำงานบน Discord ของตัวเอง

> โมเดลภาพเก่าของ OpenAI — **DALL·E 2** และ **DALL·E 3** — เป็นโมเดลรุ่นเก่า DALL·E 3 ไม่เปิดให้ใช้งานสำหรับการปรับใช้ใหม่ และฟีเจอร์เช่น `create_variation` มีแค่ใน DALL·E 2 เท่านั้น ใช้โมเดล `gpt-image` สำหรับแอปพลิเคชันใหม่

บน Microsoft Foundry, **`gpt-image-2`** เป็นโมเดลภาพรุ่นล่าสุดและมีความสามารถมากที่สุด และเป็นค่าเริ่มต้นที่แนะนำ `gpt-image-1.5` และ `gpt-image-1-mini` ก็มีพร้อมใช้งานโดยทั่วไปเช่นกัน

> **สำคัญ:** โมเดล `gpt-image` จะคืนภาพที่สร้างเป็น **base64** (`b64_json`) ไม่ใช่ URL โค้ดของคุณจะถอดรหัสสตริง base64 เป็นไบต์และบันทึกภาพ — ไม่มี URL ของภาพให้ดาวน์โหลด


## การสร้างแอปพลิเคชันสร้างภาพแรกของคุณ

แล้วต้องใช้อะไรบ้างในการสร้างแอปพลิเคชันสร้างภาพ? คุณต้องใช้ไลบรารีต่อไปนี้:

- **python-dotenv**, ขอแนะนำอย่างยิ่งให้ใช้ไลบรารีนี้เพื่อเก็บความลับของคุณในไฟล์ *.env* แยกจากโค้ด
- **openai**, ไลบรารีนี้คือสิ่งที่คุณจะใช้ในการโต้ตอบกับ OpenAI API
- **pillow**, เพื่อทำงานกับภาพใน Python

หากยังไม่ได้ทำ กรุณาทำตามคำแนะนำในหน้า [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) เพื่อสร้างแหล่งข้อมูล Microsoft Foundry และโมเดล เลือก **gpt-image-2** เป็นโมเดล (โมเดลรูปภาพ Azure OpenAI ล่าสุด; DALL·E เป็นโมเดลรุ่นเก่า)

1. สร้างไฟล์ *.env* โดยมีเนื้อหาดังนี้:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    หา ข้อมูลนี้ในพอร์ทัล Microsoft Foundry สำหรับแหล่งข้อมูลของคุณในส่วน "Deployments"



1. รวบรวมไลบรารีข้างต้นในไฟล์ที่ชื่อ *requirements.txt* ดังนี้:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. ต่อไป สร้างสภาพแวดล้อมเสมือนและติดตั้งไลบรารี:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> สำหรับ Windows ให้ใช้คำสั่งต่อไปนี้เพื่อสร้างและเปิดใช้งานสภาพแวดล้อมเสมือน:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. เพิ่มโค้ดต่อไปนี้ในไฟล์ชื่อ *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # นำเข้า dotenv
    dotenv.load_dotenv()

    # กำหนดค่าไคลเอนต์ Azure OpenAI (Microsoft Foundry)
    # โมเดลภาพต้องใช้เวอร์ชัน API ล่าสุด — ตรวจสอบเอกสาร Foundry สำหรับเวอร์ชันที่โมเดลของคุณต้องการ
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # สร้างภาพโดยใช้ API การสร้างภาพ
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ป้อนข้อความคำสั่งของคุณที่นี่
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # เช่น "gpt-image-2"
        )
        # กำหนดไดเรกทอรีสำหรับเก็บภาพ
        image_dir = os.path.join(os.curdir, 'images')

        # หากไดเรกทอรีไม่มีอยู่ ให้สร้างมันขึ้นมา
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # เริ่มต้นเส้นทางภาพ (หมายเหตุ: นามสกุลไฟล์ควรเป็น png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # โมเดล gpt-image ส่งคืนภาพในรูปแบบ base64 (b64_json) ไม่ใช่ URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # แสดงภาพในโปรแกรมดูภาพเริ่มต้น
        image = Image.open(image_path)
        image.show()

    # ดักจับข้อยกเว้น
    except BadRequestError as err:
        print(err)

    ```

มาอธิบายโค้ดนี้กัน:

- อย่างแรก เราทำการนำเข้าห้องสมุดที่เราต้องการ รวมถึงห้องสมุด OpenAI, ห้องสมุด dotenv, โมดูล base64, และห้องสมุด Pillow

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- ต่อมา เราทำการโหลดตัวแปรสภาพแวดล้อมจากไฟล์ *.env*

    ```python
    # นำเข้า dotenv
    dotenv.load_dotenv()
    ```

- หลังจากนั้น เราตั้งค่าลูกค้า Azure OpenAI (Microsoft Foundry)

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- ต่อไป เราสร้างภาพ:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ป้อนข้อความพร้อมท์ของคุณที่นี่
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    โมเดล `gpt-image` จะคืนภาพมาเป็นสตริง **base64** ใน `data[0].b64_json` เราทำการถอดรหัสเป็นไบต์และเขียนลงไฟล์ — ไม่มี URL สำหรับดาวน์โหลด

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- สุดท้าย เราเปิดภาพและใช้โปรแกรมดูภาพมาตรฐานแสดงภาพนั้น:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### รายละเอียดเพิ่มเติมเกี่ยวกับการสร้างภาพ

มาดูพารามิเตอร์ของ `images.generate` กัน:

- **prompt** คือข้อความพิมพ์ที่ใช้สร้างภาพ ที่นี่คือ "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils"
- **size** คือขนาดของภาพที่สร้าง (เช่น `1024x1024`, `1536x1024`, `1024x1536` หรือ `"auto"`)
- **n** คือจำนวนภาพที่สร้าง ที่นี่เราสร้างหนึ่งภาพ
- **model** คือชื่อดีพลอยเมนต์โมเดลภาพของคุณ (เช่น `gpt-image-2`)

> โมเดลภาพจะไม่มีพารามิเตอร์ `temperature` — นั่นคือการควบคุมการสร้างข้อความ สำหรับการได้ภาพหลากหลาย ให้เรียก API อีกครั้ง ถ้าต้องการลดความหลากหลาย ให้ระบุ prompt ให้ชัดเจนขึ้น

## ความสามารถเพิ่มเติมในการสร้างภาพ

คุณได้เห็นวิธีสร้างภาพด้วยโค้ด Python เพียงไม่กี่บรรทัด โมเดล `gpt-image` ยังสามารถ **แก้ไข** ภาพที่มีอยู่ได้ โดยให้ภาพต้นฉบับ, **mask** (อ็อปชันที่ระบุพื้นที่ที่ต้องการเปลี่ยนแปลง), และ prompt คุณสามารถเปลี่ยนแปลงส่วนหนึ่งของภาพ — เช่น เพิ่มหมวกให้กระต่ายของเรา

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# การแก้ไขจะถูกส่งกลับมาในรูปแบบ base64 ด้วยเช่นกัน
edited_image = base64.b64decode(response.data[0].b64_json)
```

ภาพฐานมีแค่กระต่ายเท่านั้น; ภาพสุดท้ายมีกระต่ายใส่หมวกแล้ว


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
